In [ ]:
#@title CONFIG — edit this cell only
PROJECT_NAME = "korea_test"
DRIVE_ROOT = "/content/drive/MyDrive/AI_Video_Studio"
ZIP_FILENAME = "source.zip"
SCRIPT_FILENAME = "script.txt"
CONFIG_FILENAME = None  # Optional: project_config.json in the input folder
TEST_MODE = False
FORCE_REGENERATE_VOICE = False  # set True once when migrating an existing project from 4 to 8 steps
FORCE_REBUILD = False
VOICE_NUM_STEP = 8  # production balanced default; 4 for smoke test, 16 optional quality
VOICE_TIMEOUT_SECONDS = 20 * 60
COPY_MODEL_TO_LOCAL = True  # faster inference; model is persisted in Drive
REPO_URL = "https://github.com/justmeireneng/Video-Animation-Release.git"
REPO_BRANCH = "main"
TEST_USE_EXISTING_VOICE = False  # diagnostic only
TEST_SKIP_VOICE = False  # diagnostic only; produces PARTIAL_PASS


# AI Video Studio — Colab runner

Upload `source.zip` and `script.txt` to the Drive input folder, edit the CONFIG cell above if needed, then choose **Runtime → Run all**. The notebook calls the repository's canonical `build-video` command; it does not duplicate the render pipeline.

## 1. Mount Drive and prepare folders

The notebook never deletes Drive files. Work files are rendered on Colab's local disk, then the final MP4, report, project state, and caches are copied back to Drive.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, time, re, platform
from datetime import datetime, timezone
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError('Google Drive mount failed. Re-run this cell and authorize Drive access.') from exc

DRIVE_ROOT_PATH = Path(DRIVE_ROOT)
DRIVE_INPUT = DRIVE_ROOT_PATH / 'input'
DRIVE_OUTPUT = DRIVE_ROOT_PATH / 'output' / PROJECT_NAME
DRIVE_MODELS = DRIVE_ROOT_PATH / 'models'
DRIVE_CACHE = DRIVE_ROOT_PATH / 'cache'
DRIVE_PROJECTS = DRIVE_ROOT_PATH / 'projects'
DRIVE_LOGS = DRIVE_ROOT_PATH / 'logs'
for folder in (DRIVE_INPUT, DRIVE_OUTPUT, DRIVE_MODELS, DRIVE_CACHE, DRIVE_PROJECTS, DRIVE_LOGS):
    folder.mkdir(parents=True, exist_ok=True)

ZIP_DRIVE = DRIVE_INPUT / ZIP_FILENAME
SCRIPT_DRIVE = DRIVE_INPUT / SCRIPT_FILENAME
if not ZIP_DRIVE.is_file() or not SCRIPT_DRIVE.is_file():
    missing = []
    if not ZIP_DRIVE.is_file(): missing.append(f'ZIP: {ZIP_DRIVE}')
    if not SCRIPT_DRIVE.is_file(): missing.append(f'SCRIPT: {SCRIPT_DRIVE}')
    raise FileNotFoundError('INPUT FILE NOT FOUND\nExpected:\n' + '\n'.join(missing))
print('Drive ready:', DRIVE_ROOT_PATH)
print('ZIP:', ZIP_DRIVE)
print('Script:', SCRIPT_DRIVE)

## 2. Clone/update the repository and install pinned dependencies

The repository lockfile remains the source of truth. OmniVoice is pinned to the version used by the local provider (`0.2.1`).

In [ ]:
def run_cmd(command, cwd=None, check=True, quiet=False):
    command = [str(item) for item in command]
    result = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=quiet, check=False)
    if check and result.returncode != 0:
        detail = (result.stderr or result.stdout or '').strip()[-2000:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{detail}")
    return result

WORK_ROOT = Path('/content/ai-video-work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / 'repo'
if not (REPO_DIR / '.git').is_dir():
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    print('Repository already present; updating with a fast-forward pull.')
    run_cmd(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)

setup_started = time.perf_counter()
run_cmd(['apt-get', 'update', '-qq'])
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    run_cmd(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
try:
    import omnivoice  # noqa: F401
except ImportError:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'omnivoice==0.2.1'])

pnpm_version = run_cmd(['pnpm', '--version'], quiet=True).stdout.strip() if shutil.which('pnpm') else ''
if not pnpm_version.startswith('9.'):
    run_cmd(['npm', 'install', '-g', 'pnpm@9'])
    pnpm_version = run_cmd(['pnpm', '--version'], quiet=True).stdout.strip()
remotion_dir = REPO_DIR / 'remotion'
frozen = subprocess.run(['pnpm', 'install', '--frozen-lockfile'], cwd=str(remotion_dir), text=True, capture_output=True, check=False)
if frozen.returncode != 0:
    detail = (frozen.stderr or frozen.stdout or '').strip()[-4000:]
    print('Frozen lockfile install failed; retrying in the ephemeral Colab checkout.')
    print(detail)
    shutil.rmtree(remotion_dir / 'node_modules', ignore_errors=True)
    run_cmd(['pnpm', 'install', '--no-frozen-lockfile'], cwd=remotion_dir)
print('pnpm:', pnpm_version)
setup_seconds = round(time.perf_counter() - setup_started, 3)
print(f'Environment ready in {setup_seconds}s')
print('Python:', platform.python_version())
print('Node:', run_cmd(['node', '--version'], quiet=True).stdout.strip())
print('pnpm:', run_cmd(['pnpm', '--version'], quiet=True).stdout.strip())
print('ffmpeg:', run_cmd(['ffmpeg', '-version'], quiet=True).stdout.splitlines()[0])

## 3. Detect GPU and prepare persistent OmniVoice model cache

The notebook reports the actual PyTorch CUDA state and exports `OMNIVOICE_DEVICE` for the core provider. It does not assume that a visible GPU is usable.

In [ ]:
gpu_name = 'none'
nvidia = subprocess.run(['bash', '-lc', 'nvidia-smi --query-gpu=name --format=csv,noheader'], text=True, capture_output=True, check=False)
if nvidia.returncode == 0 and nvidia.stdout.strip():
    gpu_name = nvidia.stdout.strip().splitlines()[0]
try:
    import torch
    pytorch_cuda = bool(torch.cuda.is_available())
    cuda_version = torch.version.cuda or 'none'
except Exception:
    pytorch_cuda, cuda_version = False, 'unavailable'
OMNIVOICE_DEVICE = 'cuda' if pytorch_cuda and gpu_name != 'none' else 'cpu'
os.environ['OMNIVOICE_DEVICE'] = OMNIVOICE_DEVICE
os.environ['HF_HOME'] = str(DRIVE_MODELS / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(DRIVE_MODELS / 'huggingface' / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(DRIVE_MODELS / 'huggingface' / 'transformers')
os.environ['OMNIVOICE_PYTHON'] = sys.executable
model_setup_started = time.perf_counter()
from huggingface_hub import snapshot_download
snapshot = snapshot_download('k2-fsa/OmniVoice', cache_dir=str(DRIVE_MODELS / 'huggingface'))
local_model = WORK_ROOT / 'omnivoice-model'
if COPY_MODEL_TO_LOCAL:
    model_marker = local_model / '.ready'
    if not model_marker.is_file():
        if local_model.exists(): shutil.rmtree(local_model)
        shutil.copytree(snapshot, local_model)
        model_marker.touch()
    os.environ['OMNIVOICE_MODEL_PATH'] = str(local_model)
else:
    os.environ['OMNIVOICE_MODEL_PATH'] = str(snapshot)
model_setup_seconds = round(time.perf_counter() - model_setup_started, 3)
print('GPU_DETECTED:', 'yes' if gpu_name != 'none' else 'no')
print('GPU_NAME:', gpu_name)
print('CUDA_VERSION:', cuda_version)
print('PYTORCH_CUDA_AVAILABLE:', pytorch_cuda)
print('OMNIVOICE_DEVICE:', OMNIVOICE_DEVICE)
print('OmniVoice model path:', os.environ['OMNIVOICE_MODEL_PATH'])
print(f'Model setup/cache ready in {model_setup_seconds}s')

## 4. Copy inputs/state to local Colab storage and run the canonical build

Existing project state and per-scene voice cache are copied from Drive before the build. The core command remains `python app.py build-video`; the notebook only supplies paths and environment.

In [ ]:
WORK_INPUT = WORK_ROOT / 'input'
WORK_INPUT.mkdir(parents=True, exist_ok=True)
zip_local = WORK_INPUT / ZIP_FILENAME
script_local = WORK_INPUT / SCRIPT_FILENAME
shutil.copy2(ZIP_DRIVE, zip_local)
shutil.copy2(SCRIPT_DRIVE, script_local)
project_id = re.sub(r'[^a-z0-9]+', '-', PROJECT_NAME.lower()).strip('-')[:64] or 'untitled-project'
DRIVE_OUTPUT = DRIVE_ROOT_PATH / 'output' / project_id
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
work_project = REPO_DIR / 'projects' / project_id
drive_project = DRIVE_PROJECTS / project_id
if drive_project.is_dir():
    shutil.copytree(drive_project, work_project, dirs_exist_ok=True)
if FORCE_REGENERATE_VOICE and work_project.exists():
    for relative in ('voice/cache', 'voice/narration', 'metadata/logs/voice-stage.json', 'metadata/logs/voice-progress.json'):
        target = work_project / relative
        if target.is_dir(): shutil.rmtree(target)
        elif target.exists(): target.unlink()
if FORCE_REBUILD and work_project.exists():
    for relative in ('render', 'build_report.json'):
        target = work_project / relative
        if target.is_dir(): shutil.rmtree(target)
        elif target.exists(): target.unlink()
config_local = None
if CONFIG_FILENAME:
    config_drive = DRIVE_INPUT / CONFIG_FILENAME
    if not config_drive.is_file(): raise FileNotFoundError(f'Optional config not found: {config_drive}')
    config_local = WORK_INPUT / CONFIG_FILENAME
    shutil.copy2(config_drive, config_local)

command = [sys.executable, 'app.py', 'build-video', '--zip', str(zip_local), '--script', str(script_local), '--project', project_id, '--voice-timeout-seconds', str(VOICE_TIMEOUT_SECONDS)]
if VOICE_NUM_STEP is not None: command += ['--voice-num-step', str(VOICE_NUM_STEP)]
if config_local: command += ['--config', str(config_local)]
if (TEST_USE_EXISTING_VOICE or TEST_SKIP_VOICE) and not TEST_MODE:
    raise ValueError('Diagnostic voice flags require TEST_MODE=True.')
if TEST_USE_EXISTING_VOICE: command += ['--use-existing-voice']
if TEST_SKIP_VOICE: command += ['--skip-voice']
if TEST_MODE:
    print('TEST MODE: all scenes remain enabled; voice changes only through the explicit TEST_* flags.')
print('================================')
print('AI VIDEO STUDIO — COLAB BUILD')
print('================================')
print('[1/8] Environment ready')
print('[2/8] Input validation passed')
print('[3/8] Import and scene mapping')
build_started = time.perf_counter()
process = subprocess.Popen(command, cwd=str(REPO_DIR), env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for raw_line in process.stdout:
    line = raw_line.rstrip()
    captured.append(line)
    if any(token in line for token in ('Voice stage:', 'Generating voice', 'Voice ready', 'BUILD FAILED')):
        print(line)
return_code = process.wait()
build_seconds = round(time.perf_counter() - build_started, 3)
(DRIVE_LOGS / project_id).mkdir(parents=True, exist_ok=True)
(DRIVE_LOGS / project_id / 'build-console.log').write_text('\n'.join(captured) + '\n', encoding='utf-8')
if return_code != 0:
    print('Core build failed. The project state/report were still persisted to Drive.')
else:
    print('Core build command completed.')

## 5. Persist output, validate, and show the final video

A final file is copied to Drive only when the core report says `FULL_PASS`. Test modes remain explicitly `PARTIAL_PASS`.

In [ ]:
report_path = work_project / 'build_report.json'
if work_project.exists():
    shutil.copytree(work_project, drive_project, dirs_exist_ok=True)
if not report_path.is_file():
    raise FileNotFoundError(f'Build report was not produced: {report_path}')
report = json.loads(report_path.read_text(encoding='utf-8'))
drive_report = DRIVE_OUTPUT / 'build_report.json'
drive_report.parent.mkdir(parents=True, exist_ok=True)
run_report = {
    'project': project_id, 'gpu': gpu_name, 'omnivoice_device': OMNIVOICE_DEVICE,
    'environment_setup_seconds': setup_seconds, 'model_setup_seconds': model_setup_seconds,
    'build_seconds': build_seconds, 'core_status': report.get('status'),
    'voice_seconds': (report.get('stages', {}).get('voice') or {}).get('elapsed_seconds'),
    'preview_seconds': (report.get('stages', {}).get('preview') or {}).get('elapsed_seconds'),
    'final_seconds': (report.get('stages', {}).get('final') or {}).get('elapsed_seconds'),
    'created_at': datetime.now(timezone.utc).isoformat(),
}
if report.get('status') == 'FULL_PASS':
    final_local = work_project / 'render' / 'final' / 'final.mp4'
    if not final_local.is_file(): raise FileNotFoundError(f'Validated final is missing: {final_local}')
    final_drive = DRIVE_OUTPUT / 'final.mp4'
    shutil.copy2(final_local, final_drive)
    shutil.copy2(report_path, drive_report)
    run_report['final_path'] = str(final_drive)
    print('BUILD COMPLETE')
    print('Project:', project_id)
    print('Scenes:', report.get('scene_count'))
    print('GPU:', gpu_name)
    print('OmniVoice device:', report.get('voice', {}).get('device'))
    print('Voice cache hits:', report.get('voice', {}).get('cache_hits'))
    print('Final:', final_drive)
    print('Status: FULL_PASS')
else:
    shutil.copy2(report_path, drive_report)
    run_report['final_path'] = None
    print('BUILD STATUS:', report.get('status'))
    print('No final.mp4 was copied because validation did not return FULL_PASS.')
(DRIVE_OUTPUT / 'colab_run_report.json').write_text(json.dumps(run_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Build report:', drive_report)
print('Timing:', json.dumps(run_report, ensure_ascii=False, indent=2))

if report.get('status') == 'FULL_PASS':
    from IPython.display import Video, display
    display(Video(str(DRIVE_OUTPUT / 'final.mp4'), embed=False))